# Entity Action BC v1 — Training + Evaluation with Battle Diagnostics

**Notebook Version**: 1.1  
**Purpose**: End-to-end training of `entity_action_bc_v1` with integrated battle analysis

Phases:
1. **Setup** — GPU, deps, data
2. **Train** — entity model with sequence auxiliary head
3. **Inspect** — model summary, artifacts, training metadata
4. **Evaluate** — load model, generate battle logs
5. **Diagnostics** — win rate, action accuracy, tactical errors

Reference: `docs/BATTLE_ANALYSIS_FRAMEWORK.md`

## Setup: GPU, Dependencies, and Data

In [ ]:
# ── 1. Verify GPU ──────────────────────────────────────────────────────────
import subprocess

gpu_available = False
try:
    r = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
        capture_output=True, text=True, timeout=5
    )
    if r.returncode == 0:
        print('GPU:', r.stdout.strip())
        gpu_available = True
except (FileNotFoundError, subprocess.TimeoutExpired):
    pass

import tensorflow as tf
print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print('TF GPUs:', [g.name for g in gpus])
    gpu_available = True
else:
    print('TF: CPU only')

if not gpu_available:
    print()
    print('WARNING: No GPU detected.')
    print('Colab: Runtime -> Change runtime type -> T4 GPU, then restart.')

In [ ]:
# ── 2. Clone repo ──────────────────────────────────────────────────────────
import os

REPO_URL = 'https://github.com/AlterProgramming/Pokemon-Showdown-Sim.git'
BRANCH   = 'main'
REPO_DIR = '/content/Pokemon-Showdown-Sim'

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
!git log --oneline -3

In [ ]:
# ── 3. Install dependencies ────────────────────────────────────────────────
import importlib, tensorflow as tf
print('TF:', tf.__version__)

!pip install -q kagglehub

try:
    import keras
    if int(keras.__version__.split('.')[0]) < 3:
        print('Upgrading keras to 3.x...')
        !pip install -q 'keras>=3.0'
        importlib.invalidate_caches()
    else:
        print('keras', keras.__version__, 'OK')
except ImportError:
    !pip install -q 'keras>=3.0'

print('Done. If first run: Runtime -> Restart session, then re-run from cell 1.')

In [ ]:
# ── 4. Download battle data ────────────────────────────────────────────────
import kagglehub, os, glob

DATASET = 'thephilliplin/pokemon-showdown-battles-gen9-randbats'
print('Downloading', DATASET, '...')
data_path = kagglehub.dataset_download(DATASET)

json_files = glob.glob(os.path.join(data_path, '*.json'))
print(f'Dataset path : {data_path}')
print(f'JSON files   : {len(json_files):,}')

## Training Configuration

In [ ]:
# ── 5. Training configuration ──────────────────────────────────────────────
import os
from datetime import datetime

# Unique model name per run — avoids 'Refusing to overwrite' on re-runs
_ts = datetime.now().strftime('%Y%m%d_%H%M')
MODEL_NAME = f'entity_action_bc_v1_{_ts}'
OUTPUT_DIR = '/content/training_artifacts'
RUN_ID     = MODEL_NAME

MAX_BATTLES   = 100      # increase to 5000 for production
VAL_RATIO     = 0.2
EPOCHS        = 30
BATCH_SIZE    = 256
LEARNING_RATE = 0.001
PATIENCE      = 3
HIDDEN_DIM    = 256
DEPTH         = 3
DROPOUT       = 0.1

PREDICT_TURN_OUTCOME  = True
PREDICT_VALUE         = True
PREDICT_TURN_SEQUENCE = True

TRANSITION_WEIGHT    = 0.25
VALUE_WEIGHT         = 0.25
SEQUENCE_WEIGHT      = 0.1
SEQUENCE_HIDDEN_DIM  = 128
MAX_SEQ_LEN          = 32

POLICY_RETURN_WEIGHTING    = 'exp'
POLICY_RETURN_WEIGHT_SCALE = 0.75
POLICY_RETURN_WEIGHT_MIN   = 0.25
POLICY_RETURN_WEIGHT_MAX   = 4.0

os.makedirs(OUTPUT_DIR, exist_ok=True)

print('=' * 70)
print('TRAINING CONFIGURATION')
print('=' * 70)
print(f'  Model name   : {MODEL_NAME}')
print(f'  Output dir   : {OUTPUT_DIR}')
print(f'  Max battles  : {MAX_BATTLES}')
print(f'  Epochs       : {EPOCHS}  (early stop patience={PATIENCE})')
print(f'  Batch size   : {BATCH_SIZE}   LR={LEARNING_RATE}')
print(f'  Architecture : hidden={HIDDEN_DIM}  depth={DEPTH}  dropout={DROPOUT}')
print()
print('  Auxiliary heads:')
print(f'    transition : {PREDICT_TURN_OUTCOME}  (w={TRANSITION_WEIGHT})')
print(f'    value      : {PREDICT_VALUE}  (w={VALUE_WEIGHT})')
print(f'    sequence   : {PREDICT_TURN_SEQUENCE}  (w={SEQUENCE_WEIGHT}  lstm={SEQUENCE_HIDDEN_DIM}  maxlen={MAX_SEQ_LEN})')
print('=' * 70)


## Phase 1: Training

In [ ]:
# ── 6. Train (epoch-level output only) ────────────────────────────────────
import subprocess, sys, os, re

REPO_DIR = '/content/Pokemon-Showdown-Sim'
env = os.environ.copy()
env['PYTHONPATH'] = REPO_DIR + os.pathsep + env.get('PYTHONPATH', '')

cmd = [
    sys.executable, '-u', 'train_entity_action.py', data_path,
    '--model-name', MODEL_NAME,
    '--output-dir', OUTPUT_DIR,
    '--max-battles', str(MAX_BATTLES),
    '--val-ratio',   str(VAL_RATIO),
    '--epochs',      str(EPOCHS),
    '--batch-size',  str(BATCH_SIZE),
    '--learning-rate', str(LEARNING_RATE),
    '--patience',    str(PATIENCE),
    '--hidden-dim',  str(HIDDEN_DIM),
    '--depth',       str(DEPTH),
    '--dropout',     str(DROPOUT),
]
if PREDICT_TURN_OUTCOME:
    cmd += ['--predict-turn-outcome', '--transition-weight', str(TRANSITION_WEIGHT)]
if PREDICT_VALUE:
    cmd += ['--predict-value', '--value-weight', str(VALUE_WEIGHT)]
if PREDICT_TURN_SEQUENCE:
    cmd += [
        '--predict-turn-sequence',
        '--sequence-weight',     str(SEQUENCE_WEIGHT),
        '--sequence-hidden-dim', str(SEQUENCE_HIDDEN_DIM),
        '--max-seq-len',         str(MAX_SEQ_LEN),
    ]
cmd += [
    '--policy-return-weighting',    POLICY_RETURN_WEIGHTING,
    '--policy-return-weight-scale', str(POLICY_RETURN_WEIGHT_SCALE),
    '--policy-return-weight-min',   str(POLICY_RETURN_WEIGHT_MIN),
    '--policy-return-weight-max',   str(POLICY_RETURN_WEIGHT_MAX),
    '--seed', '42',
]

# Patterns — setup markers and epoch summaries
SETUP_KWS = [
    'entity_training_start', 'entity_data_source', 'entity_dataset',
    'entity_policy_weighting', 'entity_initialization', 'entity_transfer',
    'training_history', 'Restoring', 'early stopping',
]
EPOCH_HDR  = re.compile(r'Epoch\s+\d+/\d+')
METRIC_RE  = re.compile(r'loss|accuracy|mae|brier', re.IGNORECASE)
ERROR_RE   = re.compile(r'error|traceback|exception|warning|failed|killed', re.IGNORECASE)

print('Command:', ' '.join(cmd[:4]), '...')
print('=' * 70)

proc = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, cwd=REPO_DIR, env=env,
)

current_epoch_lines = []
suppressed_lines    = []   # kept so we can dump on failure

for raw_line in proc.stdout:
    line = raw_line.rstrip()
    if not line:
        continue

    # ── Always show: errors / tracebacks / warnings ──────────────────────
    if ERROR_RE.search(line):
        print(line)
        suppressed_lines = []   # reset suppression buffer on error
        continue

    # ── Always show: setup / status markers ──────────────────────────────
    if any(kw in line for kw in SETUP_KWS):
        print(line)
        continue

    # ── Epoch header ──────────────────────────────────────────────────────
    if EPOCH_HDR.match(line):
        print()
        print(line)
        current_epoch_lines = []
        suppressed_lines.append(line)
        continue

    # ── Keras metric line — keep latest per epoch ─────────────────────────
    if METRIC_RE.search(line):
        current_epoch_lines.append(line)
        suppressed_lines.append(line)
        continue

    # ── Separator after epoch — flush last metrics line ───────────────────
    if current_epoch_lines and re.match(r'^\s*$', line):
        print(current_epoch_lines[-1])
        current_epoch_lines = []
        continue

    # All other lines — buffer silently
    suppressed_lines.append(line)

# Flush trailing metrics
if current_epoch_lines:
    print(current_epoch_lines[-1])

proc.wait()
print()
print('=' * 70)
print(f'Exit code: {proc.returncode}')

# ── On failure: dump the last N suppressed lines so we can diagnose ───────
if proc.returncode != 0:
    print()
    print('TRAINING FAILED — last output lines:')
    print('-' * 70)
    for l in suppressed_lines[-60:]:
        print(l)
    print('-' * 70)
    raise RuntimeError(f'Training subprocess exited with code {proc.returncode}')


## Phase 2: Inspect Artifacts & Model Summary

In [ ]:
# ── 7. Artifacts listing and training metadata ─────────────────────────────
import os, json

artifacts = sorted(os.listdir(OUTPUT_DIR))

print('=' * 70)
print('ARTIFACTS')
print('=' * 70)
for f in artifacts:
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    size_str = f'{size/(1024*1024):.2f} MB' if size > 1024*1024 else f'{size/1024:.1f} KB'
    print(f'  {f:55s}  {size_str:>10}')

meta_files = [f for f in artifacts if f.startswith('training_metadata')]
if meta_files:
    meta = json.load(open(os.path.join(OUTPUT_DIR, meta_files[0])))
    print()
    print('TRAINING METADATA')
    print('-' * 70)
    for k in [
        'model_name', 'training_regime', 'objective_set',
        'epochs_completed', 'epochs_requested',
        'train_examples', 'val_examples',
        'sequence_vocab_size', 'sequence_hidden_dim', 'max_seq_len',
        'sequence_weight', 'transition_weight', 'value_weight',
    ]:
        if k in meta:
            print(f'  {k:30s}: {meta[k]}')
    if 'entity_token_vocab_sizes' in meta:
        print()
        print('  Entity token vocab sizes:')
        for name, sz in meta['entity_token_vocab_sizes'].items():
            print(f'    {name:20s}: {sz:,}')

In [ ]:
# ── 8. Model summary ─────────────────────────────────────────────────────
# Rebuilds the architecture from training metadata and loads only the weights.
# This avoids all Lambda-deserialization issues regardless of when the model
# was saved.
import os, sys, json
sys.path.insert(0, '/content/Pokemon-Showdown-Sim')
from tensorflow import keras
from core.EntityModelV1 import build_entity_action_models
from core.StateVectorization import turn_outcome_dim

artifacts = sorted(os.listdir(OUTPUT_DIR))
meta_files         = [f for f in artifacts if f.startswith('training_metadata')]
policy_files       = [f for f in artifacts if f.endswith('.keras') and 'training_model' not in f]
training_model_files = [f for f in artifacts if f.startswith('training_model')]
seq_vocab_files    = [f for f in artifacts if 'sequence_vocab' in f]
policy_vocab_files = [f for f in artifacts if 'policy_vocab' in f and f.endswith('.json')]

if not meta_files or not policy_files:
    print('Artifacts missing — check training exit code'); raise SystemExit

meta = json.load(open(os.path.join(OUTPUT_DIR, meta_files[0])))
print(f'Metadata  : {meta_files[0]}')

# Prefer the training model artifact when sequence/value/transition heads were on,
# because the policy-only artifact lacks those heads.
use_training_model = bool(training_model_files)
weights_file = training_model_files[0] if use_training_model else policy_files[0]
weights_path = os.path.join(OUTPUT_DIR, weights_file)
print(f'Weights   : {weights_file}')

# Rebuild architecture exactly as it was trained
predict_seq        = 'sequence_auxiliary' in meta.get('objective_set', [])
predict_transition = 'transition' in meta.get('objective_set', [])
predict_value      = 'value' in meta.get('objective_set', [])

training_model, policy_model, _ = build_entity_action_models(
    vocab_sizes         = meta['entity_token_vocab_sizes'],
    num_policy_classes  = meta['num_action_classes'],
    hidden_dim          = meta['hidden_dim'],
    depth               = meta['depth'],
    dropout             = meta['dropout'],
    learning_rate       = meta['learning_rate'],
    token_embed_dim     = meta.get('token_embed_dim', 24),
    transition_dim      = turn_outcome_dim() if predict_transition else None,
    action_context_vocab_size = meta.get('num_action_context_classes') or None,
    action_embed_dim    = meta.get('action_embed_dim', 16),
    transition_hidden_dim = meta.get('transition_hidden_dim', meta['hidden_dim']),
    transition_weight   = meta.get('transition_weight', 0.25),
    predict_value       = predict_value,
    value_hidden_dim    = meta.get('value_hidden_dim') or None,
    value_weight        = meta.get('value_weight', 0.25),
    predict_sequence    = predict_seq,
    sequence_vocab_size = meta.get('sequence_vocab_size') or None,
    sequence_hidden_dim = meta.get('sequence_hidden_dim', 128),
    sequence_weight     = meta.get('sequence_weight', 0.1),
    max_seq_len         = meta.get('max_seq_len', 32),
)

# Load weights into the freshly-built model — no Lambda deserialisation involved
display_model = training_model if use_training_model else policy_model
display_model.load_weights(weights_path)
print()
print('=' * 70)
print(f'MODEL SUMMARY  ({weights_file})')
print('=' * 70)
display_model.summary()
print('=' * 70)

if policy_vocab_files:
    pv = json.load(open(os.path.join(OUTPUT_DIR, policy_vocab_files[0])))
    print(f'\nPolicy vocab  : {len(pv)} actions')
if seq_vocab_files:
    sv = json.load(open(os.path.join(OUTPUT_DIR, seq_vocab_files[0])))
    print(f'Sequence vocab: {len(sv)} event tokens')


## Phase 3: Battle Analysis Framework (Diagnostics)

In [ ]:
# ── 9. BattleAnalyzer utilities ────────────────────────────────────────────
import numpy as np
from typing import Dict, List, Any

class BattleAnalyzer:
    """Battle analysis utilities — see docs/BATTLE_ANALYSIS_FRAMEWORK.md."""

    @staticmethod
    def compute_win_rate(battle_logs: List[Dict[str, Any]]) -> Dict[str, Any]:
        """Win rate + Wilson-score 95% CI."""
        wins  = sum(1 for bl in battle_logs if bl.get('metadata', {}).get('p1_won', False))
        total = len(battle_logs)
        if total == 0:
            return {'win_rate': 0.0, 'wins': 0, 'total_battles': 0}
        p, z  = wins / total, 1.96
        denom = 1 + z**2 / total
        centre = (p + z**2 / (2 * total)) / denom
        spread = np.sqrt(p * (1 - p) / total + z**2 / (4 * total**2)) / denom
        return {
            'win_rate': float(p), 'wins': wins, 'total_battles': total,
            'ci_lower': float(max(0.0, centre - z * spread)),
            'ci_upper': float(min(1.0, centre + z * spread)),
        }

    @staticmethod
    def compute_action_accuracy(battle_logs: List[Dict[str, Any]]) -> Dict[str, Any]:
        """Move and switch accuracy across all decision points."""
        move_total = move_ok = switch_total = switch_ok = 0
        for bl in battle_logs:
            for dp in bl.get('decision_points', []):
                legal  = dp.get('legal_actions', [])
                chosen = dp.get('action_chosen', '')
                if any(a.startswith('move:') for a in legal):
                    move_total += 1
                    if chosen.startswith('move:'):
                        move_ok += 1
                if any(a.startswith('switch:') for a in legal):
                    switch_total += 1
                    if chosen.startswith('switch:'):
                        switch_ok += 1
        return {
            'move_accuracy':   float(move_ok   / move_total)   if move_total   else 0.0,
            'move_turns':      move_total,
            'switch_accuracy': float(switch_ok / switch_total) if switch_total else 0.0,
            'switch_turns':    switch_total,
        }

    @staticmethod
    def classify_tactical_errors(battle_logs: List[Dict[str, Any]]) -> Dict[str, List]:
        """Classify tactical errors — see BATTLE_ANALYSIS_FRAMEWORK Part 6."""
        errors: Dict[str, List] = {
            'repeated_switch': [], 'status_wasted': [], 'wasted_setup': [],
            'consecutive_failures': [],
        }
        for bl in battle_logs:
            bid = bl.get('metadata', {}).get('battle_id', '?')
            dps = bl.get('decision_points', [])
            switches  = [d for d in dps if d.get('action_chosen', '').startswith('switch:')]
            for i in range(len(switches) - 1):
                if (switches[i]['action_chosen'] == switches[i+1]['action_chosen']
                        and switches[i+1].get('turn', 0) - switches[i].get('turn', 0) <= 5):
                    errors['repeated_switch'].append({'battle': bid, 'turn': switches[i].get('turn'), 'action': switches[i]['action_chosen']})
            for dp in dps:
                outcome = dp.get('move_outcome', '')
                if outcome == 'setup_wasted':
                    errors['wasted_setup'].append({'battle': bid, 'turn': dp.get('turn'), 'move': dp.get('action_chosen')})
                elif outcome == 'no_effect':
                    errors['status_wasted'].append({'battle': bid, 'turn': dp.get('turn'), 'move': dp.get('action_chosen')})
            failures = [d for d in dps if d.get('move_outcome') not in ('success', None)]
            for i in range(len(failures) - 2):
                if (failures[i+2].get('turn', 0) - failures[i].get('turn', 0) <= 3
                        and failures[i].get('player') == failures[i+1].get('player')):
                    errors['consecutive_failures'].append({'battle': bid, 'turns': (failures[i].get('turn'), failures[i+2].get('turn'))})
        return errors

print('BattleAnalyzer ready')

In [ ]:
# ── 10. Run diagnostics on sample battle logs ──────────────────────────────
# In production these come from evaluate_model.py; shown here with sample data.
import os, json
from datetime import datetime

EVAL_DIR      = '/content/evaluation_runs'
EVAL_RUN_DIR  = os.path.join(EVAL_DIR, RUN_ID)
DIAGNOSTICS_DIR = os.path.join(EVAL_RUN_DIR, 'diagnostics')
os.makedirs(DIAGNOSTICS_DIR, exist_ok=True)

sample_logs = [
    {'metadata': {'battle_id': 'g001', 'p1_won': True,  'duration_turns': 45},
     'decision_points': [
         {'turn': 1, 'player': 'p1', 'legal_actions': ['move:tackle', 'switch:pikachu'], 'action_chosen': 'move:tackle', 'move_outcome': 'success'},
         {'turn': 2, 'player': 'p1', 'legal_actions': ['move:tackle', 'switch:pikachu'], 'action_chosen': 'move:tackle', 'move_outcome': 'success'},
     ]},
    {'metadata': {'battle_id': 'g002', 'p1_won': False, 'duration_turns': 30},
     'decision_points': [
         {'turn': 1, 'player': 'p1', 'legal_actions': ['move:tackle', 'switch:pikachu'], 'action_chosen': 'switch:pikachu', 'move_outcome': 'success'},
         {'turn': 2, 'player': 'p1', 'legal_actions': ['move:tackle', 'switch:pikachu'], 'action_chosen': 'switch:pikachu', 'move_outcome': 'no_effect'},
         {'turn': 3, 'player': 'p1', 'legal_actions': ['move:tackle'],                  'action_chosen': 'move:tackle',    'move_outcome': 'no_effect'},
     ]},
]

win_stats    = BattleAnalyzer.compute_win_rate(sample_logs)
action_stats = BattleAnalyzer.compute_action_accuracy(sample_logs)
errors       = BattleAnalyzer.classify_tactical_errors(sample_logs)

print('=' * 70)
print('BATTLE DIAGNOSTICS')
print('=' * 70)

print('\n1. WIN RATE')
print(f'   {win_stats["wins"]}/{win_stats["total_battles"]} battles won  ({win_stats["win_rate"]:.1%})')
print(f'   95% CI: [{win_stats["ci_lower"]:.1%}, {win_stats["ci_upper"]:.1%}]')

print('\n2. ACTION ACCURACY')
print(f'   Move   : {action_stats["move_accuracy"]:.1%}  ({action_stats["move_turns"]} turns)')
print(f'   Switch : {action_stats["switch_accuracy"]:.1%}  ({action_stats["switch_turns"]} turns)')

print('\n3. TACTICAL ERRORS')
for err_type, err_list in errors.items():
    print(f'   {err_type:25s}: {len(err_list)}')
    for e in err_list[:2]:
        print(f'     {e}')

print('\n' + '=' * 70)

# Persist reports
for fname, payload in [
    ('win_rate_summary.json',  {'run_id': RUN_ID, **win_stats}),
    ('action_accuracy.json',   {'run_id': RUN_ID, **action_stats}),
    ('tactical_errors.json',   {'run_id': RUN_ID, 'counts': {k: len(v) for k, v in errors.items()}, 'detail': errors}),
]:
    path = os.path.join(DIAGNOSTICS_DIR, fname)
    json.dump(payload, open(path, 'w'), indent=2)
    print(f'Saved: {path}')

## Summary & Download

In [ ]:
# ── 11. Summary ────────────────────────────────────────────────────────────
import os

print('=' * 70)
print('RUN SUMMARY')
print('=' * 70)
print(f'  Run ID         : {RUN_ID}')
print(f'  Training dir   : {OUTPUT_DIR}')
print(f'  Diagnostics dir: {DIAGNOSTICS_DIR}')
print()
print('  Next steps for production:')
print('    1. Set MAX_BATTLES=5000 and re-run')
print('    2. Run evaluate_model.py to generate real battle logs')
print('    3. Re-run diagnostics cell with real logs')
print('    4. See docs/BATTLE_ANALYSIS_FRAMEWORK.md for full reference')
print('=' * 70)

In [ ]:
# ── 12. Download artifacts (Colab only) ───────────────────────────────────
try:
    from google.colab import files
    import os

    to_download = [
        (OUTPUT_DIR,      [f for f in artifacts if f.endswith('.keras') or 'vocab' in f or 'metadata' in f]),
        (DIAGNOSTICS_DIR, ['win_rate_summary.json', 'action_accuracy.json', 'tactical_errors.json']),
    ]
    for directory, file_list in to_download:
        for fname in file_list:
            path = os.path.join(directory, fname)
            if os.path.exists(path):
                print(f'Downloading {fname} ...')
                files.download(path)
    print('Done.')
except ImportError:
    print('Not on Colab — artifacts at:', OUTPUT_DIR)